
# Session 2: Regression with scikit-learn  
## Theory + intuition + demos version

**Course theme:** Predicting numerical outcomes with machine learning  
**Suggested duration:** 6 hours  
**Level:** Beginner  
**Main library:** scikit-learn

---

## Why this version?

This notebook upgrades the regression tutorial by adding:

- more **theory**
- more **math intuition**
- more **small demos**
- more explanation of **how the models behave**

It still avoids more advanced workflow topics like grid search and feature engineering, so students can stay focused on the main regression ideas first.

---

## Learning goals

By the end of this session, students should be able to:

- explain what regression is
- explain what a regression model is trying to optimize
- understand the difference between:
  - Linear Regression
  - Ridge Regression
  - Lasso Regression
  - Elastic Net
- interpret basic regression metrics
- read actual-vs-predicted and residual plots
- understand coefficient shrinkage
- understand why regularization matters
- use scikit-learn to train and compare regression models



## Suggested 6-hour teaching flow

### Hour 1
- What is regression?
- Examples from business
- What a model is trying to learn
- One-feature demo

### Hour 2
- Linear regression theory
- Loss function
- Fitting a line
- Predictions and residuals

### Hour 3
- Train/test split
- Metrics: MAE, MSE, RMSE, R²
- Actual vs predicted
- Residual plot

### Hour 4
- Coefficients and interpretation
- Multicollinearity intuition
- Why ordinary least squares can become unstable

### Hour 5
- Ridge, Lasso, Elastic Net
- Penalty terms
- Coefficient shrinkage demos
- Effect of alpha

### Hour 6
- Practical modeling on a dataset
- Model comparison
- Exercises
- Wrap-up


In [ ]:

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline



# Part 1: What is regression?

A **regression problem** is a machine learning problem where the target is a **number**.

Examples:
- predict house price
- predict monthly sales
- predict salary
- predict hospital cost
- predict delivery time
- predict energy usage

In this session, we will predict a numeric target called **price**.

---

## A very important distinction

### Classification
Predicts a category  
Examples:
- spam / not spam
- churn / not churn
- approve / reject

### Regression
Predicts a number  
Examples:
- 250000
- 18.7
- 54.2



# Part 2: The core idea of linear regression

Suppose we only have one feature, such as square footage.

A very simple regression model might look like:

\[
\hat{y} = b_0 + b_1 x
\]

Where:

- \(\hat{y}\) = predicted value
- \(b_0\) = intercept
- \(b_1\) = slope / coefficient
- \(x\) = feature value

This is just the equation of a line.

---

## What is the model trying to do?

It tries to choose the "best" line so that predictions are close to the real values.

The difference between an actual value and a predicted value is called a **residual**:

\[
\text{residual} = y - \hat{y}
\]

A common idea is to make the total squared residuals as small as possible:

\[
\text{Loss} = \sum (y - \hat{y})^2
\]

This is the key idea behind **ordinary least squares** linear regression.

---

## Why square the errors?

Because:
- positive and negative errors should not cancel out
- large errors should be punished more strongly



## Demo 1: One-feature regression by eye

Let's create a tiny synthetic example so students can see that a regression line is just trying to follow the trend in the data.


In [ ]:

# Tiny synthetic example
x_demo = np.array([1, 2, 3, 4, 5, 6])
y_demo = np.array([2, 4, 5, 4, 5, 7])

plt.figure(figsize=(6, 4))
plt.scatter(x_demo, y_demo, s=60)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Small Example: Points with a Rough Linear Trend")
plt.show()


In [ ]:

# Fit a linear regression line to the tiny example
X_demo = x_demo.reshape(-1, 1)

small_lr = LinearRegression()
small_lr.fit(X_demo, y_demo)

y_demo_pred = small_lr.predict(X_demo)

print("Intercept:", small_lr.intercept_)
print("Slope:", small_lr.coef_[0])

plt.figure(figsize=(6, 4))
plt.scatter(x_demo, y_demo, s=60, label="Actual points")
plt.plot(x_demo, y_demo_pred, linestyle="--", label="Fitted line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression Fits a Best-Fit Line")
plt.legend()
plt.show()



## What students should notice

- The line does not pass through every point
- The model is not trying to memorize the data
- It is trying to find a line that gives a **good overall fit**



## Demo 2: Residuals in a tiny example

Residuals are the vertical distances from the points to the line:

\[
\text{residual} = y - \hat{y}
\]

If the prediction is too low, the residual is positive.  
If the prediction is too high, the residual is negative.


In [ ]:

# Calculate residuals for the tiny example
residuals_demo = y_demo - y_demo_pred

residual_table = pd.DataFrame({
    "x": x_demo,
    "actual_y": y_demo,
    "predicted_y": np.round(y_demo_pred, 2),
    "residual": np.round(residuals_demo, 2)
})

residual_table


In [ ]:

# Plot the line and show residuals as vertical segments
plt.figure(figsize=(6, 4))
plt.scatter(x_demo, y_demo, s=60)
plt.plot(x_demo, y_demo_pred, linestyle="--")

for i in range(len(x_demo)):
    plt.plot([x_demo[i], x_demo[i]], [y_demo[i], y_demo_pred[i]])

plt.xlabel("x")
plt.ylabel("y")
plt.title("Residuals = Vertical Distance from Point to Line")
plt.show()



# Part 3: From toy data to a practical dataset

We now move to a larger housing-style dataset.

**Target:** `price`

We will use this dataset to build and compare regression models.


In [ ]:

# Load the dataset
df = pd.read_csv("/mnt/data/housing_regression_practice.csv")

# Show the first few rows
df.head()


In [ ]:

# Dataset shape and data types
print("Rows and columns:", df.shape)
print()
print(df.dtypes)


In [ ]:

# Summary statistics
df.describe()



## Features and target

In machine learning:

- **X** = input variables / features
- **y** = target variable

We will keep the first part simple and use only the numeric features.


In [ ]:

# Define X and y
X = df.drop(columns="price")
y = df["price"]

# Keep only numeric features for beginner simplicity
X_num = X.select_dtypes(include=["int64", "float64"])

print("Numeric feature names:")
print(X_num.columns.tolist())
print()
print("X_num shape:", X_num.shape)
print("y shape:", y.shape)



## Why do we split into train and test sets?

We want to know how well the model works on **new, unseen data**.

- **Training set**: used to learn patterns
- **Test set**: used to evaluate the learned model

This helps us avoid fooling ourselves.


In [ ]:

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_num, y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)



# Part 4: Linear Regression in practice

The scikit-learn model for ordinary least squares is:

- `LinearRegression()`

This model tries to minimize:

\[
\sum (y - \hat{y})^2
\]

That means:
- it wants predictions close to the true values
- it punishes large errors more heavily


In [ ]:

# Create and train a linear regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

print("Intercept:", linear_model.intercept_)
print("Number of coefficients:", len(linear_model.coef_))


In [ ]:

# Predict on training and test data
train_pred = linear_model.predict(X_train)
test_pred = linear_model.predict(X_test)

# Show a few predictions
comparison_df = pd.DataFrame({
    "actual": y_test.iloc[:10].values,
    "predicted": np.round(test_pred[:10], 0).astype(int)
})

comparison_df



# Part 5: Regression metrics

Different metrics tell different stories.

---

## 1. MAE

\[
MAE = \frac{1}{n}\sum |y - \hat{y}|
\]

Interpretation:
- average absolute error
- easy to explain

---

## 2. MSE

\[
MSE = \frac{1}{n}\sum (y - \hat{y})^2
\]

Interpretation:
- average squared error
- larger mistakes hurt more

---

## 3. RMSE

\[
RMSE = \sqrt{MSE}
\]

Interpretation:
- easier than MSE because it is in the original unit of the target

---

## 4. R²

\[
R^2 = 1 - \frac{\sum (y-\hat{y})^2}{\sum (y-\bar{y})^2}
\]

Interpretation:
- how much variation in the target is explained by the model
- higher is better


In [ ]:

# Helper function to print regression metrics
def regression_report(y_true, y_pred, label="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)

    print(f"--- {label} ---")
    print(f"MAE : {mae:,.2f}")
    print(f"MSE : {mse:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R^2 : {r2:.4f}")
    print()

regression_report(y_train, train_pred, label="Linear Regression (Train)")
regression_report(y_test, test_pred, label="Linear Regression (Test)")



## How should students interpret train vs test metrics?

- If training performance is much better than test performance, overfitting may be happening
- If both are poor, the model may be too simple or the problem may be difficult



# Part 6: Visual evaluation

Plots often reveal things that summary metrics hide.


In [ ]:

# Actual vs predicted plot
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.6)

min_val = y_test.min()
max_val = y_test.max()

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted (Test Set)")
plt.show()



## Why do we draw the diagonal line?

This line represents:

\[
\text{predicted} = \text{actual}
\]

So:
- points on the line = perfect predictions
- points above the line = over-predictions
- points below the line = under-predictions


In [ ]:

# Residual plot
residuals = y_test - test_pred

plt.figure(figsize=(8, 4))
plt.scatter(test_pred, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.show()



## How to read the residual plot

A good residual plot usually shows:
- points scattered around 0
- no strong curve
- no obvious funnel pattern

If strong patterns appear, the model may be missing structure in the data.



# Part 7: Coefficients and interpretation

For linear regression, the prediction formula is:

\[
\hat{y} = b_0 + b_1x_1 + b_2x_2 + \dots + b_px_p
\]

Each coefficient tells us the direction and size of the feature's contribution.

### If a coefficient is positive:
As the feature increases, the prediction tends to increase.

### If a coefficient is negative:
As the feature increases, the prediction tends to decrease.

Important warning:

A predictive coefficient is **not automatically a causal effect**.


In [ ]:

# Create a coefficient table
coef_df = pd.DataFrame({
    "feature": X_num.columns,
    "coefficient": linear_model.coef_
}).sort_values("coefficient", ascending=False)

coef_df


In [ ]:

# Show strongest positive and negative coefficients
top_positive = coef_df.head(10)
top_negative = coef_df.tail(10).sort_values("coefficient")

print("Top positive coefficients")
display(top_positive)

print("\nTop negative coefficients")
display(top_negative)



## Discussion questions

- Which features seem to raise price?
- Which features seem to lower price?
- Do the signs look reasonable?
- Can we say a coefficient proves causation?

Important answer:
**No. Regression for prediction is not the same as causal analysis.**



# Part 8: Why ordinary least squares can struggle

Linear regression is powerful, but it can become unstable when:

- features are highly correlated
- data is noisy
- one feature can partly replace another
- coefficients become too large

This leads us to **regularization**.



## Multicollinearity intuition demo

Let's build two almost-duplicate features.

If two features contain nearly the same information, the model may have trouble deciding how much weight each one should get.


In [ ]:

# Create correlated features for intuition
rng = np.random.default_rng(0)

x1 = np.linspace(0, 10, 80)
x2 = x1 + rng.normal(0, 0.2, 80)   # almost the same as x1
y_corr = 5 + 3*x1 + rng.normal(0, 1.0, 80)

X_corr = pd.DataFrame({
    "x1": x1,
    "x2": x2
})

corr_model = LinearRegression()
corr_model.fit(X_corr, y_corr)

print("Coefficients with correlated features:")
print(pd.DataFrame({
    "feature": X_corr.columns,
    "coefficient": corr_model.coef_
}))



Even if both features are very similar, the coefficients may be split in a somewhat unstable way.  
This is one reason regularization is useful.



# Part 9: Ridge Regression

Ridge Regression keeps the same idea as linear regression, but adds a penalty for large coefficients.

### Objective function

\[
\min \sum (y - \hat{y})^2 + \alpha \sum w_j^2
\]

This means the model tries to do **two things at once**:

1. fit the data well
2. keep coefficients small

---

## What does alpha do?

- small \(\alpha\) → behaves more like ordinary linear regression
- large \(\alpha\) → stronger shrinkage
- very large \(\alpha\) → coefficients become very small, model may underfit

---

## Intuition

Ridge says:

> "Fit the data well, but do not use unnecessarily large coefficients."


In [ ]:

# Train a Ridge model
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

# Make predictions
ridge_pred = ridge_model.predict(X_test)

# Evaluate
regression_report(y_test, ridge_pred, label="Ridge Regression")



## Demo: coefficient shrinkage in Ridge

We will compare Linear Regression and Ridge coefficients side by side.


In [ ]:

ridge_coef_df = pd.DataFrame({
    "feature": X_num.columns,
    "LinearRegression": linear_model.coef_,
    "Ridge": ridge_model.coef_
})

ridge_coef_df


In [ ]:

# Demo: how alpha changes Ridge behavior
ridge_alphas = [0, 0.1, 1, 10, 100]
ridge_demo_rows = []

for a in ridge_alphas:
    if a == 0:
        model = LinearRegression()
        model.fit(X_train, y_train)
        coef_size = np.abs(model.coef_).mean()
        rmse = mean_squared_error(y_test, model.predict(X_test)) ** 0.5
        ridge_demo_rows.append({
            "model": "LinearRegression",
            "alpha": 0,
            "avg_abs_coef": coef_size,
            "test_rmse": rmse
        })
    else:
        model = Ridge(alpha=a)
        model.fit(X_train, y_train)
        coef_size = np.abs(model.coef_).mean()
        rmse = mean_squared_error(y_test, model.predict(X_test)) ** 0.5
        ridge_demo_rows.append({
            "model": "Ridge",
            "alpha": a,
            "avg_abs_coef": coef_size,
            "test_rmse": rmse
        })

pd.DataFrame(ridge_demo_rows)



Students should notice:
- as alpha grows, coefficients usually shrink
- prediction error may improve or worsen depending on the problem



# Part 10: Lasso Regression

Lasso also adds a penalty, but a different one.

### Objective function

\[
\min \sum (y - \hat{y})^2 + \alpha \sum |w_j|
\]

The key difference is that Lasso uses **absolute values** instead of squares.

---

## Why is that important?

Because Lasso can shrink some coefficients to **exactly zero**.

That means:
- it simplifies the model
- it may effectively remove some features
- it can act a bit like automatic feature selection


In [ ]:

# Train a Lasso model
lasso_model = Lasso(alpha=0.1, max_iter=10000)
lasso_model.fit(X_train, y_train)

# Make predictions
lasso_pred = lasso_model.predict(X_test)

# Evaluate
regression_report(y_test, lasso_pred, label="Lasso Regression")


In [ ]:

# Compare Linear Regression and Lasso coefficients
lasso_coef_df = pd.DataFrame({
    "feature": X_num.columns,
    "LinearRegression": linear_model.coef_,
    "Lasso": lasso_model.coef_
})

lasso_coef_df


In [ ]:

# Count how many coefficients are exactly zero in Lasso
lasso_zero_count = np.sum(lasso_model.coef_ == 0)
print("Number of coefficients set to zero by Lasso:", lasso_zero_count)


In [ ]:

# Demo: effect of alpha in Lasso
lasso_alphas = [0.001, 0.01, 0.1, 1, 10]
lasso_demo_rows = []

for a in lasso_alphas:
    model = Lasso(alpha=a, max_iter=10000)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    lasso_demo_rows.append({
        "alpha": a,
        "zero_coefficients": int(np.sum(model.coef_ == 0)),
        "avg_abs_coef": np.abs(model.coef_).mean(),
        "test_rmse": mean_squared_error(y_test, pred) ** 0.5
    })

pd.DataFrame(lasso_demo_rows)



Students should notice:
- larger alpha tends to create more zeros
- Lasso can produce simpler models than Ridge



# Part 11: Elastic Net

Elastic Net combines Ridge and Lasso ideas.

### Objective function

\[
\min \sum (y - \hat{y})^2 + \alpha \left[(1-\rho)\sum w_j^2 + \rho \sum |w_j|\right]
\]

In scikit-learn:
- `alpha` controls total regularization strength
- `l1_ratio` controls the mix between Ridge-like and Lasso-like behavior

---

## Intuition

Elastic Net is useful when:
- features are correlated
- we want shrinkage
- we also want some zero coefficients


In [ ]:

# Train an Elastic Net model
elastic_model = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
elastic_model.fit(X_train, y_train)

# Make predictions
elastic_pred = elastic_model.predict(X_test)

# Evaluate
regression_report(y_test, elastic_pred, label="Elastic Net")


In [ ]:

# Compare coefficients across models
coef_compare = pd.DataFrame({
    "feature": X_num.columns,
    "LinearRegression": linear_model.coef_,
    "Ridge": ridge_model.coef_,
    "Lasso": lasso_model.coef_,
    "ElasticNet": elastic_model.coef_
})

coef_compare



# Part 12: Side-by-side model comparison

Now we compare all four models on the same test data.


In [ ]:

results = pd.DataFrame([
    {
        "model": "Linear Regression",
        "MAE": mean_absolute_error(y_test, test_pred),
        "RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
        "R2": r2_score(y_test, test_pred)
    },
    {
        "model": "Ridge",
        "MAE": mean_absolute_error(y_test, ridge_pred),
        "RMSE": mean_squared_error(y_test, ridge_pred) ** 0.5,
        "R2": r2_score(y_test, ridge_pred)
    },
    {
        "model": "Lasso",
        "MAE": mean_absolute_error(y_test, lasso_pred),
        "RMSE": mean_squared_error(y_test, lasso_pred) ** 0.5,
        "R2": r2_score(y_test, lasso_pred)
    },
    {
        "model": "Elastic Net",
        "MAE": mean_absolute_error(y_test, elastic_pred),
        "RMSE": mean_squared_error(y_test, elastic_pred) ** 0.5,
        "R2": r2_score(y_test, elastic_pred)
    }
])

results.sort_values("RMSE")



## How should students compare models?

Do not look at only one thing.

Consider:
- prediction accuracy
- model simplicity
- interpretability
- coefficient stability
- whether some coefficients become zero



# Part 13: A tiny behavior demo with an outlier

One reason squared error matters is that large errors are punished strongly.

Let's see what happens when we add an outlier to a simple dataset.


In [ ]:

# Original clean data
x_clean = np.array([1, 2, 3, 4, 5, 6]).reshape(-1, 1)
y_clean = np.array([2, 4, 6, 8, 10, 12])

# Add an outlier
x_out = np.array([1, 2, 3, 4, 5, 6, 7]).reshape(-1, 1)
y_out = np.array([2, 4, 6, 8, 10, 12, 30])

model_clean = LinearRegression()
model_clean.fit(x_clean, y_clean)

model_out = LinearRegression()
model_out.fit(x_out, y_out)

x_plot = np.linspace(1, 7, 100).reshape(-1, 1)

plt.figure(figsize=(7, 4))
plt.scatter(x_clean, y_clean, label="Clean data")
plt.scatter([7], [30], label="Outlier")
plt.plot(x_plot, model_clean.predict(x_plot), linestyle="--", label="Fit without outlier")
plt.plot(x_plot, model_out.predict(x_plot), linestyle="--", label="Fit with outlier")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression Can Be Pulled by Outliers")
plt.legend()
plt.show()



This helps students see that:
- ordinary least squares is sensitive to large squared errors
- unusual points can strongly influence the fitted line



# Part 14: Exercises

## Exercise 1
Write the linear regression prediction formula in words.

## Exercise 2
Explain the difference between:
- actual value
- predicted value
- residual

## Exercise 3
Explain why MSE punishes large errors more than MAE.

## Exercise 4
Train Ridge with:
- `alpha = 0.1`
- `alpha = 10`

What changes?

## Exercise 5
Train Lasso with:
- `alpha = 1`

How many coefficients become zero?

## Exercise 6
Use only four features:
- `sqft`
- `bedrooms`
- `bathrooms`
- `age`

Compare this smaller model to the full one.

## Exercise 7
In plain English, explain:
- Linear Regression
- Ridge
- Lasso
- Elastic Net


In [ ]:

# Starter code for Exercise 6

small_features = ["sqft", "bedrooms", "bathrooms", "age"]
X_small = df[small_features]
y_small = df["price"]

X_small_train, X_small_test, y_small_train, y_small_test = train_test_split(
    X_small, y_small, test_size=0.20, random_state=42
)

small_model = LinearRegression()
small_model.fit(X_small_train, y_small_train)

small_pred = small_model.predict(X_small_test)

regression_report(y_small_test, small_pred, label="Smaller Linear Regression Model")



# Part 15: Key takeaways

- Regression predicts a **number**
- Linear Regression fits a line or linear combination of features
- Ordinary least squares minimizes **sum of squared errors**
- Residuals are **actual - predicted**
- MAE, MSE, RMSE, and R² each tell a different story
- Ridge adds an **L2 penalty**
- Lasso adds an **L1 penalty**
- Elastic Net combines both
- Regularization helps control coefficient size and improve stability
- The goal is not just to call an API, but to understand:
  - what the model is doing
  - what it is optimizing
  - how its behavior changes



# End of notebook

A natural next session could cover:
- classification
- tree-based models
- practical tuning tips
- model validation strategy
